In [ ]:
from IPython.core.display import Image
from typing import TypedDict
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph. checkpoint.memory import InMemorySaver

from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain_deepseek import ChatDeepSeek

import sys
from loguru import logger
logger. remove()
logger.add(sys.stdout, colorize=True)

from dotenv import load_dotenv

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
    "thinking": {
    "type": "disabled"
    }
        }
)

# 构建子图
def llm_node(state: MessagesState) -> MessagesState:
    messages = state["messages"]
    logger.info("=" * 30)
    logger.info("调用子图的 llm_node 节点,当前的 messages:")
    for index, message in enumerate(messages, start=1):
        logger.opt(colors=True).info(
        "\n<cyan><bold>[ {}]</bold></cyan>\n"
        "<yellow>: </yellow><magenta>{}</magenta>\n"
        "<yellow>: </yellow><green>{}</green>",
        index,
        message.type,
        message.content
        )

    logger.info("=" * 30)
    response = model.invoke(input=messages)
    ai_msg = AIMessage(content=response.content)
    return {"messages": [ai_msg]}


builder = StateGraph(state_schema=MessagesState)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

# 【策略切换点】只需修改这一行
# subgraph =builder.compile() #per-invocation(默认):每次调用状态隔离
subgraph = builder. compile(checkpointer=True)

#

# Per-thread:历史相互干扰

# 构建父图
class OverAllstate(TypedDict):
    user_inputs: list[str] # 用户提问
    assistant_responses: list[str] # 助手回答

def call_subgraph(state: OverAllstate) -> OverAllstate:
    user_inputs = state["user_inputs"]

    assistant_responses = []
    for user_input in user_inputs:
        subgraph_response = subgraph.invoke({
            "messages": [
                SystemMessage("用最简短的话回答用户的提问"),
                HumanMessage(user_input)
        ]
        }
       )
        assistant_response = subgraph_response["messages"][-1].content
        assistant_responses.append(assistant_response)

    return {
    "assistant_responses": assistant_responses
    }


builder = StateGraph(state_schema=OverAllstate)
builder.add_node("call_subgraph", call_subgraph)
builder.add_edge(START, "call_subgraph")
builder.add_edge("call_subgraph", END)

checkpointer = InMemorySaver()
parent_graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "multi-call-same-demo"} }

first_response = parent_graph.invoke({
    "user_inputs": [
        "5*5等于几",
        "10*10等于几"]
    },
    config = config)

print("="* 30,"->第一次运行结果 <- ","="*30)
print(first_response)

second_response = parent_graph.invoke({
    "user_inputs": [
        "再加1呢？",
        "再加10000呢？"
    ]
    },
    config = config)
display(
    Image(
        parent_graph.get_graph(xray=True).draw_mermaid_png()
    )
)
print("="* 30,"->第二次运行结果 <- ","="*30)
print(second_response)


